# Gemma 3n GPU Fine-Tuning (LoRA)

This notebook fine-tunes a Gemma 3n instruct model on a **single GPU** using LoRA adapters.

## What you get
- GPU environment checks
- Local JSONL dataset loading
- Prompt formatting for supervised fine-tuning
- LoRA setup with PEFT
- Training + adapter export

> Replace `MODEL_ID` if your preferred Gemma 3n checkpoint name differs in your registry.

In [ ]:
# Install dependencies (run once per environment)
# Fix torch/torchvision compatibility first
%pip install -q -U "torch>=2.3" "torchvision>=0.18" "transformers>=4.53.0,<5.0" "tokenizers>=0.21.1" "datasets>=2.20.0" "peft>=0.15,<1.0" "trl>=0.15,<1.0" "accelerate>=1.0,<2.0" 

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.24.0 requires torch==2.9.0, but you have torch 2.10.0 which is incompatible.

[notice] A new release of pip is available

In [ ]:
%pip install timm

In [ ]:
import os
from dataclasses import dataclass

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

@dataclass
class Config:
    model_id: str = "google/gemma-3n-E2B-it"
    data_file: str = "finetuning_data.json"
    output_dir: str = "outputs/cpu-lora"
    max_seq_len: int = 32768 
    learning_rate: float = 2e-4
    num_train_epochs: int = 5
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 4
    warmup_ratio: float = 0.05

cfg = Config()
cfg

c:\Users\Avinash.Deyyam\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


Config(model_id='google/gemma-3n-e4b-it', train_file='data/train.jsonl', eval_file='data/eval.jsonl', output_dir='outputs/gemma3n-lora', max_seq_len=1024, learning_rate=0.0002, num_train_epochs=3, per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=8, warmup_ratio=0.05)

In [ ]:
# GPU sanity checks
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not available. Use a GPU runtime to fine-tune efficiently.")

print("gpu count:", torch.cuda.device_count())
print("gpu name:", torch.cuda.get_device_name(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

torch: 2.10.0+cpu
cuda available: False


RuntimeError: CUDA GPU not available. Use a GPU runtime to fine-tune efficiently.

## Dataset format
Create local files:
- `data/train.jsonl`
- `data/eval.jsonl`

Each line should contain:
```json
{"instruction": "What is wrong with this valve?", "response": "Likely seal wear near the stem."}
```

You can add extra fields, but `instruction` and `response` are required by this notebook formatter.

In [ ]:
# Load dataset from local JSON lines file
if not os.path.exists(cfg.data_file):
    raise FileNotFoundError(f"Missing data file: {cfg.data_file}")

raw = load_dataset("json", data_files={"train": cfg.data_file})["train"]
split = raw.train_test_split(test_size=0.1, seed=42)
dataset = {"train": split["train"], "eval": split["test"]}
len(dataset["train"]), len(dataset["eval"])

In [ ]:
# Prompt formatter (uses instruction/context/response)
def format_example(example):
    instruction = str(example.get("instruction", "")).strip()
    context = str(example.get("context", "")).strip()
    response = str(example.get("response", "")).strip()

    if context:
        user_block = f"Instruction: {instruction}\nContext: {context}"
    else:
        user_block = f"Instruction: {instruction}"

    text = (
        "<bos><start_of_turn>user\n"
        + user_block
        + "<end_of_turn>\n<start_of_turn>model\n"
        + response
        + "<end_of_turn>"
    )
    return {"text": text}

dataset["train"] = dataset["train"].map(format_example)
dataset["eval"] = dataset["eval"].map(format_example)
dataset["train"][0]["text"][:300]

In [ ]:
# LoRA + tokenizer/model loading

tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "up_proj", "down_proj", "gate_proj"],
)

model

In [ ]:
## Trainer setup
sft_config = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type="cosine",
    warmup_ratio=cfg.warmup_ratio,
    logging_steps=10,
    save_steps=20,
    eval_steps=20,
    eval_strategy="steps",
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="none",
)

# Rename 'text' column to expected format
def formatting_func(examples):
    return examples["text"]

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=sft_config,
    peft_config=peft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    formatting_func=formatting_func,
)
trainer

In [ ]:
# Start training
train_result = trainer.train()
train_result

In [ ]:
# Save adapter + tokenizer
trainer.model.save_pretrained(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print(f"Saved fine-tuned adapter to: {cfg.output_dir}")

In [ ]:
# Optional quick inference check
from transformers import pipeline

generator = pipeline(
    task="text-generation",
    model=cfg.output_dir,
    tokenizer=cfg.output_dir,
    device_map="auto",
)

prompt = "<bos><start_of_turn>user\nHow should I inspect a noisy valve in the field?<end_of_turn>\n<start_of_turn>model\n"
out = generator(prompt, max_new_tokens=128, do_sample=False)
print(out[0]["generated_text"])